# Python程序设计（项目实战） · 工单P2.1《Pandas数据处理管道》 — 教师参考版
第3周 | 参考实现可直接运行验证 | 答案请勿下发学生

In [ ]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())


## 参考实现 `p21`（含依赖导入）

In [ ]:
import sys
import numpy as np  # 通用科学计算


def p21():
    import pandas as pd

    df = pd.read_csv("p21_raw.csv")
    log = [f"加载 {len(df)} 行"]
    log.append(f"temp缺失 {df.temp.isna().sum()}")

    df["temp"] = pd.to_numeric(df.temp, errors="coerce")     # 类型错→NaN
    parts = []
    for fmt in ("%Y-%m-%d %H:%M:%S", "%m/%d/%Y %H:%M", "%Y%m%d%H%M%S"):
        parts.append(pd.to_datetime(df.ts, format=fmt, errors="coerce"))
    ts = parts[0]
    for p in parts[1:]:
        ts = ts.combine_first(p)                            # 三格式统一
    df["ts"] = ts

    df["temp"] = df.temp.interpolate()                       # 决策: 时序插值
    q1, q3 = df.temp.quantile([.25, .75])
    iqr = q3 - q1
    biz = df.temp > 120                                     # 业务阈值
    df = df[~((df.temp < q1 - 1.5 * iqr) | (df.temp > q3 + 1.5 * iqr) | biz)]
    df = df.drop_duplicates(subset=["device_id", "ts"])
    df.to_csv("clean.csv", index=False)

    s = (40 * (1 - df.isna().mean().mean())
         + 30 * df.temp.between(20, 120).mean()
         + 20 * df.device_id.str.match(r"DEV-\d{2}").mean()
         + 10)
    print("\n".join(log))
    print(f"清洗后 {len(df)} 行, 质量评分 {round(s, 1)}")


## 运行环境(如需)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '学生代码框架/p21_pipeline.py', '--peek'], check=True)
print('样例脏数据已生成: p21_raw.csv')

## 执行与验证

In [ ]:
p21()